# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` values as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review the available record sets, their `@id`s, fields, and columns present in the dataset.


In [ ]:
# List all record sets and display their information (@id, name, etc.)
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name} | {field.data_type}")
    print(f"  Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"    - {col.id}: {col.name}")
    print('-'*50)


Let's preview the first few records in each record set, using their `@id`.

In [ ]:
# Display a preview of the first 2 records for each record set
for rs in record_sets:
    print(f"RecordSet @id: {rs.id} | name: {rs.name}\nSample record(s):")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs.id)):
            pprint.pprint(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f"Could not read records for {rs.id}: {e}")
    print('-'*60)


## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. All operations use the record set, column, and field `@id` as per the Croissant schema.


In [ ]:
# List record set @ids, as discovered above
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}.")
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}:\n{df.columns.tolist()}\n")
        print(df.head(2))
    except Exception as e:
        print(f"Error loading DataFrame for {record_set_id}: {e}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field/column references use their `@id` as provided in the metadata.


In [ ]:
# Select a record set with numeric fields for demonstration (use the first one for which this works)
an_example_set = None
numeric_field_id = None

for rs in record_sets:
    df = dataframes.get(rs.id)
    if df is not None:
        for field in rs.fields:
            # Only consider columns that exist in the DataFrame
            if field.id in df.columns and field.data_type in ['Integer', 'Float', 'Number']:
                # Found a viable numeric field
                an_example_set = rs.id
                numeric_field_id = field.id
                print(f"Using RecordSet {an_example_set} and numeric field {numeric_field_id}.")
                break
        if an_example_set:
            break

if not an_example_set or not numeric_field_id:
    print("No numeric field found in any record set for EDA.")
else:
    # Proceed with EDA
    df = dataframes[an_example_set]
    # Remove missing/NaN
    df_numeric = df[[numeric_field_id]].dropna().copy()
    # Cast to numeric, if not already
    df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors='coerce')
    # Simple threshold filtering: Above mean value
    threshold = df_numeric[numeric_field_id].mean()
    filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a non-numeric field (first string field available)
    group_field_id = None
    for field in rs.fields:
        if field.id != numeric_field_id and field.id in df.columns:
            # Assume first non-numeric and non-null field for grouping
            if df[field.id].dtype == 'O' or field.data_type == 'Text':
                group_field_id = field.id
                break
    if group_field_id is not None:
        print(f"\nGrouping by {group_field_id}:\n")
        grouped_df = filtered_df.groupby(df[group_field_id]).mean(numeric_only=True)
        print(grouped_df[[f"{numeric_field_id}_normalized"]].head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if an_example_set and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=dataframes[an_example_set], x=numeric_field_id, kde=True)
    plt.title(f'Distribution of {numeric_field_id} for RecordSet {an_example_set}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        # Boxplot by group if there are enough groups
        plt.figure(figsize=(12, 5))
        sns.boxplot(data=dataframes[an_example_set], x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id} in {an_example_set}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use `mlcroissant` to:
- Load metadata and discover record sets and fields (referencing all by their `@id` per FAIR principles).
- Extract data from one or more record sets for analysis.
- Apply basic EDA, including filtering, normalization, and grouping using schema-conformant references.
- Visualize data distributions and groupings.

All data entity IDs, fields, and transformations were referenced and documented using their Croissant schema `@id`.

**Next steps**: You can further analyze the dataset, engineer new features, or train statistical/ML models according to your specific research questions.